# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The metadata object as a Python object
metadata = dataset.metadata

# Print the dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Identify all available record sets in the dataset
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # Some schemas use recordSet
    record_sets = getattr(metadata, 'recordSet', [])

print("Record sets in the dataset:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name'] if 'name' in rs else 'Unnamed RecordSet'}")

# For demo, pick the first record set (if available)
main_record_set_id = None
if record_sets:
    main_record_set_id = record_sets[0]['@id']

if main_record_set_id:
    print(f"\nFields for record set {main_record_set_id}:\n")
    fields = record_sets[0]['field'] if 'field' in record_sets[0] else []
    for f in fields:
        print(f"- {f['@id']}: {f.get('name', 'Unnamed field')} ({f.get('dataType', '')})")
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all record set IDs
record_set_ids = []
for rs in record_sets:
    record_set_ids.append(rs['@id'])
print("Record set @ids:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records with columns: {list(df.columns)}\n")
    except Exception as e:
        print(f"  Failed to load records for {record_set_id}: {e}")

# Preview the first table if available
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Columns in principal record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Identify a numeric field by @id in the main record set
if record_sets:
    # Try to pick a likely numeric field, e.g., one with dataType Float, Integer, or Number
    field_objs = record_sets[0]['field'] if 'field' in record_sets[0] else []
    numeric_field_obj = None
    for f in field_objs:
        dtype = f.get('dataType', '')
        if dtype in ['schema:Float', 'schema:Integer', 'schema:Number', 'Float', 'Integer', 'Number']:
            numeric_field_obj = f
            break
    if numeric_field_obj is not None:
        numeric_field_id = numeric_field_obj['@id']
        print(f"Using numeric field: {numeric_field_id}")
    else:
        numeric_field_id = dataframes[main_record_set_id].select_dtypes(include=[np.number]).columns[0]
        print(f"Defaulted to numeric column: {numeric_field_id}")
else:
    numeric_field_id = None

filtered_df = dataframes[main_record_set_id]

if numeric_field_id:
    # Apply a numeric filter: e.g., > mean
    threshold = filtered_df[numeric_field_id].mean() if numeric_field_id in filtered_df else 0
    if numeric_field_id in filtered_df:
        filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} records")
        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical/text field
    group_field_id = None
    for f in field_objs:
        dtype = f.get('dataType', '')
        if dtype in ['Text', 'schema:Text', 'schema:Keyword', 'Keyword']:
            group_field_id = f['@id']
            break
    # Group, if a grouping field exists
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and filtered_df.shape[0] > 0:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If we have a grouping field for a boxplot, use it
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} distribution by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to load, inspect, and analyze the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library. We reviewed available record sets and fields by their `@id`, extracted the main record set as a dataframe, performed basic numeric filtering and normalization, and visualized the data distribution. This workflow can be adapted to other Croissant-compliant datasets for reproducible and transparent data exploration.